In [2]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import sys
from pathlib import Path

import cupy as cp
import numpy as np
from scipy.optimize import Bounds, minimize

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils.PSF_helpers import *
from utils.Zernike_helpers import *


ModuleNotFoundError: No module named 'cupy'

In [ ]:
N_order = 3              # 3 for 3P, 2 for 2P
lambd = 1.3e-3           # Wavelength [mm]
n = 1.333                # Refractive index
k = 2 * n * np.pi / lambd
num_apt = 1.05           # Numerical aperture
focal = 7.2              # Focal length of objective (Olympus) [mm]
mag = 4                  # Magnification rate from input to objective
w_0 = 3.5                # mm
alpha = np.arcsin(num_apt / n)

L_ffp = 0.01             # 10 microns [mm]
grid_ffp = 512
grid = Centered_Square_Grid(L_ffp, grid_ffp, 0)

grid_bfp = grid_ffp
L_bfp = (lambd * focal * grid_bfp) / L_ffp
microscope = Microscope(N_order, lambd, n, num_apt, focal, mag, w_0, L_bfp, grid_bfp)


In [ ]:
f = cp.asarray(np.load("images/usaf_resolution.npy"), dtype=cp.float64)


In [ ]:
def forward_PSF(microscope, grid, aberration):
    _, _, psf = microscope.compute_PSF_jax(grid, aberration)
    psf = cp.asarray(psf, dtype=cp.float64)
    return psf / cp.sum(psf)


def circular_convolve(f, psf):
    return cp.real(cp.fft.ifft2(cp.fft.fft2(f) * cp.fft.fft2(cp.fft.ifftshift(psf))))


def forward_image(microscope, grid, f, aberration):
    psf = forward_PSF(microscope, grid, aberration)
    return circular_convolve(f, psf)


def add_gaussian_noise(image, SNR, rng):
    std_dev_noise = cp.std(image) / cp.sqrt(cp.asarray(SNR, dtype=image.dtype))
    return image + rng.normal(0, std_dev_noise, image.shape, dtype=image.dtype)


In [ ]:
SNR = 1000
rng_np = np.random.default_rng(15)
rng_cp = cp.random.default_rng(15)

true_aberration = generate_johnson_aberration(1.0, alpha, rng_np)

bias_modes = [[-2, 2], [0, 2], [2, 2], [0, 4], [-4, 4], [4, 4]]
bias_strength = 0.5
a_stack = [EmptyAberration()] + [Aberration([m], [bias_strength]) for m in bias_modes]

d_stack = cp.stack([
    forward_image(microscope, grid, f, true_aberration + a)
    for a in a_stack
], axis=0)
d_stack = add_gaussian_noise(d_stack, SNR, rng_cp)


In [ ]:
def objective(microscope, grid, D_stack, a_proposed, a_stack, gamma):
    S_stack = cp.stack([
        cp.fft.fft2(forward_PSF(microscope, grid, a_proposed + a))
        for a in a_stack
    ], axis=0)

    first_term = cp.sum(cp.abs(D_stack) ** 2)
    numerator = cp.abs(cp.sum(cp.conj(S_stack) * D_stack, axis=0)) ** 2
    denominator = gamma + cp.sum(cp.abs(S_stack) ** 2, axis=0)

    return float(cp.asnumpy(cp.real(first_term - cp.sum(numerator / denominator))))


In [ ]:
def optimize(microscope, grid, d_stack, a_stack, modes_corrected, gamma):
    D_stack = cp.fft.fft2(cp.asarray(d_stack), axes=(-2, -1))
    iter_counter = {"n": 0}

    def single_arg_objective(c_guess):
        return objective(
            microscope,
            grid,
            D_stack,
            Aberration(modes_corrected, c_guess),
            a_stack,
            gamma,
        )

    def callback(c_guess):
        iter_counter["n"] += 1
        J = single_arg_objective(c_guess)
        print(f"iter {iter_counter['n']:4d}: J = {J:.6e}")
        print(f"    c = {c_guess}")

    options = {
        "disp": True,
        "return_all": True,
        "maxiter": 1000,
        "xatol": 1e-8,
        "fatol": 1e-12,
    }

    return minimize(
        single_arg_objective,
        x0=np.zeros(len(modes_corrected), dtype=np.float64),
        bounds=Bounds(-1, 1),
        method="L-BFGS-B",
        callback=callback,
        options=options,
    )


In [ ]:
modes_corrected = get_johnson_modes()
result = optimize(microscope, grid, d_stack, a_stack, modes_corrected, 1e-6)
result
